In [ ]:
import pandas as pd

# 대기오염 데이터
df_api = pd.read_csv('/content/drive/MyDrive/DS teamproj/air_quality_2024_sample.csv')

# season 계산 함수
def get_season(msrdt):
    month = int(str(msrdt)[4:6])  # YYYYMMDDHH → MM
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "fall"

# df_final: 이전에 저장한 station + park 정보
df_station_parks = pd.read_csv("/content/drive/MyDrive/DS teamproj/data/station_nearest_3parks_vector.csv")

# 필요한 칼럼 선택 및 병합 준비
df_api_filtered = df_api[['MSRSTE_NM', 'MSRDT', 'PM10']].copy()
df_api_filtered.rename(columns={'MSRSTE_NM': 'station_name', 'MSRDT': 'ymdt', 'PM10': 'pm10'}, inplace=True)

# 계절 추가
df_api_filtered['season'] = df_api_filtered['ymdt'].apply(get_season)

# ID 부여
df_api_filtered['id'] = range(1, len(df_api_filtered) + 1)

# 측정소별 도시숲 정보 병합
df_merged = pd.merge(df_api_filtered, df_station_parks, on='station_name', how='left')

# 최종 순서 정렬 (원하는 순서로 열 정렬)
cols = ['id', 'station_name', 'ymdt', 'season'] + \
       [f'park{i}_{attr}' for i in range(1, 4) for attr in ['name', 'dir_sin', 'dir_cos', 'distance', 'area']] + \
       ['pm10']

df_final_output = df_merged[cols]

# 저장
df_final_output.to_csv("/content/drive/MyDrive/DS teamproj/final_pm10_with_urbanforest.csv", index=False)
